# oai

> Read, write, and build Codex session rollouts

In [ ]:
#| default_exp oai

Codex keeps every conversation as a thread: a rollout file under `CODEX_HOME/sessions`, and an app-server that owns thread creation, history injection, resume, naming, and forks. Rollouts are append-only logs rather than ready-made conversations. They include model-visible Responses API items alongside turn context, UI events, world state, and native compaction records.

A synthetic history injected through app-server resumes like lived history. That makes threads useful as editable templates and worked examples, while leaving Codex responsible for its own storage.

In [ ]:
#| export
import base64, json, os, re, shutil, uuid
from pathlib import Path
from datetime import datetime, timezone
from openai_codex import AsyncCodex, CodexConfig
from openai_codex.api import AsyncThread
from openai_codex.generated.v2_all import ThreadInjectItemsResponse
from fastcore.utils import *
from fastllm.openai_responses import denorm_msgs
from fastllm.chat import Msg, Part, PartType, hist2fmt, data_url
from llmsurgery.hist import dlg2chat
from llmsurgery.dialog import *
from llmsurgery.compact import *
import json5

In [ ]:
from collections import Counter
from fastcore.test import *
from fastllm.chat import tool_dtls_tag, fmt2hist
from importlib.resources import files
import tempfile

## Where sessions live

Codex partitions rollouts by date below `CODEX_HOME/sessions`. The thread UUID is the final component of the filename. Codex passes `CODEX_THREAD_ID` to command processes it starts, so code in those processes can resolve its own rollout directly. A configured MCP server is long-lived and does not inherit the id for each thread.

In [ ]:
#| export
CODEX_HOME = Path.home()/'.codex'

def cur_thread():
    "The current Codex thread id, when running inside Codex"
    return os.environ.get('CODEX_THREAD_ID')

`rollout_file` resolves a UUID without assuming which date partition contains it.

In [ ]:
#| export
def rollout_file(
    thread_id, # Thread UUID
    codex_home=CODEX_HOME, # Codex home containing sessions
):
    "The persisted rollout for `thread_id`, if it exists"
    return first((Path(codex_home)/'sessions').rglob(f'*{thread_id}.jsonl'))

In [ ]:
test_eq(rollout_file('missing-thread', tempfile.mkdtemp()), None)

`load_recs` is the raw JSONL boundary. It preserves every rollout record, including events and superseded response history.

In [ ]:
#| export
def load_recs(path):
    "Rollout records read directly from JSONL `path`"
    return L(dict2obj(json.loads(line)) for line in Path(path).read_text().splitlines())

`load_rollout` combines current-thread resolution with the raw reader.

In [ ]:
#| export
def load_rollout(
    thread_id=None, # Thread UUID; `cur_thread()` if None
    codex_home=CODEX_HOME, # Codex home containing sessions
):
    "The records of a persisted Codex thread"
    thread_id = thread_id or cur_thread()
    path = rollout_file(thread_id, codex_home)
    if not path: raise FileNotFoundError(f'No rollout for thread {thread_id!r}')
    return load_recs(path)

A command run outside Codex has no `CODEX_THREAD_ID`. `project_thread` supplies the matching fallback used by `codexdojo -r`: the newest rollout whose session metadata names the project directory. Code running in the MCP server can use the same fallback when choosing the newest project thread is safe.

In [ ]:
#| export
def project_thread(
    cwd='.', # Project directory
    codex_home=CODEX_HOME, # Codex home containing sessions
):
    "The newest `(thread_id, rollout_path)` for `cwd`"
    cwd = Path(cwd).resolve()
    paths = sorted((Path(codex_home)/'sessions').rglob('*.jsonl'),key=lambda p:p.stat().st_mtime,reverse=True)
    for path in paths:
        meta = json.loads(path.open().readline())
        if meta.get('type')!='session_meta': continue
        if Path(meta['payload']['cwd']).resolve()==cwd: return meta['payload']['id'],path
    raise FileNotFoundError(f'No Codex thread for {cwd}')

In [ ]:
with tempfile.TemporaryDirectory() as td:
    home = Path(td)
    sessions = home/'sessions'
    sessions.mkdir()
    proj = home/'project'
    proj.mkdir()
    path = sessions/'rollout-thread-x.jsonl'
    path.write_text(json.dumps(dict(type='session_meta',payload=dict(id='thread-x',cwd=str(proj))))+'\n')
    newer = sessions/'rollout-newer.jsonl'
    newer.write_text('{}\n')
    mt = path.stat().st_mtime+1
    os.utime(newer, (mt,mt))
    test_eq(project_thread(proj,home),('thread-x',path))

## Creating dummy data

The examples use rollouts made by the installed app-server rather than hand-written envelopes. A source thread was created in a temporary Codex home, injected with ordinary and custom tool calls, closed, resumed by another app-server, and forked. The checked-in files preserve the exact records Codex wrote without containing a real user conversation.

In [ ]:
oai_data = Path(files('llmsurgery')/'data'/'oai')
source_path,fork_path = oai_data/'source.jsonl',oai_data/'fork.jsonl'
source_recs,fork_recs = load_recs(source_path),load_recs(fork_path)
Counter(source_recs.attrgot('type')),Counter(fork_recs.attrgot('type'))

In [ ]:
#| export
def thread_id(
    recs, # Rollout records
):
    "The owning thread id from the first session metadata record"
    return first(r.payload.get('id') for r in recs if r.get('type')=='session_meta')

In [ ]:
source_tid,fork_tid = thread_id(source_recs),thread_id(fork_recs)
test_ne(source_tid, fork_tid)
source_tid,fork_tid

### The active history

A rollout is an event log, not the history sent to the next model turn. Before compaction, its `response_item` payloads are the history. A native `compacted` record replaces everything before it with `replacement_history`; later response items extend that replacement. The latest compacted record wins.

In [ ]:
#| export
def response_items(
    recs, # Rollout records
):
    "Every Responses API item recorded in `recs`, including superseded history"
    return L(obj2dict(r.payload) for r in recs if r.get('type')=='response_item')

def split_compaction(
    recs, # Rollout records
):
    "The latest native replacement history and response items recorded after it"
    idxs = [i for i,r in enumerate(recs) if r.get('type')=='compacted']
    if not idxs: return L(),response_items(recs)
    i = idxs[-1]
    prior = L(obj2dict(x) for x in recs[i].payload.replacement_history)
    return prior,response_items(recs[i+1:])

def active_items(
    recs, # Rollout records
):
    "The Responses API history Codex reconstructs from `recs`"
    prior,new = split_compaction(recs)
    return prior+new

In [ ]:
test_eq(active_items(source_recs), response_items(source_recs))

A small real-format compaction fixture makes the replacement rule explicit. The old flux exchange remains in the file, while the replacement and its suffix are the only active items.

In [ ]:
old = [dict(type='response_item', payload=dict(type='message', role='user',
    content=[dict(type='input_text', text='Old request')]))]
replacement = [dict(type='message', role='user', content=[dict(type='input_text', text='Earlier conversation')]),
    dict(type='compaction', id='cmp_fixture', encrypted_content='encrypted fixture')]
compacted = dict(type='compacted', payload=dict(message='', replacement_history=replacement,
    window_number=1, first_window_id='w1', previous_window_id='w1', window_id='w2'))
suffix = [dict(type='response_item', payload=dict(type='message', role='user',
    content=[dict(type='input_text', text='Continue the work.')]))]
compact_recs = L(dict2obj(x) for x in [*old,compacted,*suffix])
prior,new = split_compaction(compact_recs)
test_eq([x['type'] for x in prior], ['message','compaction'])
test_eq([x['type'] for x in new], ['message'])
test_eq(active_items(compact_recs), prior+new)

### Forking

Codex forks through app-server rather than by copying JSONL. The fork gets a fresh thread ID and normal rollout metadata, while its injected history remains model-visible.

In [ ]:
source_types = [o['type'] for o in response_items(source_recs)]
fork_types = [o['type'] for o in response_items(fork_recs)]
test('custom_tool_call', source_types, in_)
test('custom_tool_call', fork_types, in_)
for typ in ('function_call','function_call_output','custom_tool_call','custom_tool_call_output'):
    test_eq(source_types.count(typ),fork_types.count(typ))
test_eq(fork_types.count('message'),source_types.count('message')+1)
source_types

## Writing a session

Responses history is a flat item list. Message items carry text, and each tool interaction is a call item followed by an output with the same `call_id`. Codex currently records host orchestration as a custom `exec` call whose input is JavaScript. The useful inner operation is normally `tools.exec_command` or an MCP call such as `tools.mcp__clikernel__execute`.

In [ ]:
#| export
def codex_msg(role, text):
    "A Responses API message item"
    typ = 'input_text' if role in ('user','developer','system') else 'output_text'
    content = text if is_listy(text) else [dict(type=typ, text=text)]
    return dict(type='message', role=role, content=content)

def codex_call(name, arguments, call_id=None):
    "A Responses API function call item"
    return dict(type='function_call', name=name, arguments=json.dumps(arguments), call_id=call_id or 'call_'+uuid.uuid4().hex)

def codex_output(call_id, output):
    "The result of a Responses API function call"
    return dict(type='function_call_output', call_id=call_id, output=output)

`parse_exec` recognizes the narrow wrappers Codex currently uses for one nested operation: stdout forwarding for `exec_command`, and image/text block forwarding for MCP results. Arbitrary JavaScript and the earlier stream wrapper remain ordinary raw `exec` calls. `exec_input` emits the current wrapper for editable nested calls.

In [ ]:
#| export
_re_exec = re.compile(r'^\s*const\s+(\w+)\s*=\s*await\s+tools\.([A-Za-z_]\w*)\((\{.*\})\);\s*(.*?)\s*$', re.S)
_re_code_template = re.compile(r'^\{\s*code\s*:\s*(String\.raw)?`(.*)`\s*\}$', re.S)

def _exec_args(raw):
    "Arguments from the JSON or template-literal forms Codex currently emits"
    try: return json5.loads(raw)
    except ValueError: pass
    m = _re_code_template.match(raw)
    if not m: return None
    raw_tag,code = m.groups()
    if '${' in code: return None
    if raw_tag: return {'code':code}
    if '\\' in code.replace(r'\\','').replace(r'\`',''): return None
    return {'code':code.replace(r'\\','\\').replace(r'\`','`')}

def parse_exec(src):
    "The logical nested tool call in a simple Codex `exec` input, or None"
    m = _re_exec.match(src)
    if not m: return None
    var,tool,raw,tail = m.groups()
    if tool=='exec_command': expected = f'text({var}.output);'
    elif tool.startswith('mcp__'): expected = f'for (const c of ({var}.content || [])) c.type === "image" ? image(c) : c.text && text(c.text);'
    else: return None
    if tail != expected: return None
    if arguments:=_exec_args(raw): return AttrDict(name='tools.'+tool, arguments=arguments)

def exec_input(name, arguments):
    "Stable `exec` JavaScript for one logical `tools.*` call"
    assert name.startswith('tools.')
    tool = name.removeprefix('tools.')
    if tool=='exec_command': result = 'text(r.output);'
    else:
        assert tool.startswith('mcp__')
        result = 'for (const c of (r.content || [])) c.type === "image" ? image(c) : c.text && text(c.text);'
    return f"const r = await tools.{tool}({json.dumps(arguments, separators=(',',':'))});\n{result}\n"

def codex_custom_call(name, arguments, call_id=None, item_id=None):
    "A Codex custom `exec` call wrapping one logical `tools.*` operation"
    call_id = call_id or 'call_'+uuid.uuid4().hex
    return dict(type='custom_tool_call', name='exec', input=exec_input(name, arguments), call_id=call_id,
        id=item_id or 'ctc_'+uuid.uuid4().hex, status='completed')

def codex_custom_output(call_id, output):
    "The result of a Codex custom tool call"
    if isinstance(output, str): output = [dict(type='input_text', text=output)]
    return dict(type='custom_tool_call_output', call_id=call_id, output=output)

In [ ]:
cc = codex_custom_call('tools.mcp__clikernel__execute', dict(code='6*7'))
test_eq(parse_exec(cc['input']).name, 'tools.mcp__clikernel__execute')
test_eq(parse_exec(cc['input']).arguments, {'code':'6*7'})
test_eq(codex_custom_output(cc['call_id'], '42')['call_id'], cc['call_id'])

exc = codex_custom_call('tools.exec_command', dict(cmd='pwd'))
test_eq(parse_exec(exc['input']).name, 'tools.exec_command')
test_eq(parse_exec(exc['input']).arguments, {'cmd':'pwd'})

captured = r'''const r = await tools.mcp__clikernel__execute({code:"dojo_start()"});
for (const c of (r.content || [])) c.type === "image" ? image(c) : c.text && text(c.text);
'''
test_eq(parse_exec(captured).name, 'tools.mcp__clikernel__execute')
test_eq(parse_exec(captured).arguments, {'code':'dojo_start()'})

templ = captured.replace('{code:"dojo_start()"}', r'{code:`x = r"\\b"`}')
test_eq(parse_exec(templ).arguments, {'code':r'x = r"\b"'})
raw = captured.replace('{code:"dojo_start()"}', r'{code:String.raw`x = "\n"`}')
test_eq(parse_exec(raw).arguments, {'code':r'x = "\n"'})

stream = r'''const r = await tools.write_stdin({session_id:17,chars:"6*7\n"});text(r.output)'''
test_eq(parse_exec(stream), None)

Captured IDs are random. `reid_items` derives item and call IDs from position and a salt, recursively updating outputs and metadata that refer to them.

In [ ]:
#| export
def _reid(prefix, key): return prefix+'_'+uuid.uuid5(uuid.NAMESPACE_URL, key).hex

def reid_items(
    items, # Responses API items
    key='', # Salt for deterministic IDs
):
    "Copies of `items` with response item and paired call IDs re-derived"
    items,ids = [obj2dict(o) for o in items],{}
    for i,o in enumerate(items):
        for field in ('id','call_id'):
            old = o.get(field)
            if old and old not in ids: ids[old] = _reid(old.partition('_')[0], f'{key}:{i}:{field}')
    def _remap(o):
        if isinstance(o, str): return ids.get(o,o)
        if isinstance(o, dict): return {k:_remap(v) for k,v in o.items()}
        if is_listy(o): return [_remap(v) for v in o]
        return o
    return L(_remap(o) for o in items)

The app-server owns thread files, and the `openai_codex` SDK is its typed client: `AsyncCodex` starts, resumes, reads, names, and forks threads. `codex_client` pins it to the PATH `codex` binary so the SDK and CLI share one rollout format, and redirects `CODEX_HOME` for tests. The one operation llmsurgery needs that lacks a first-class SDK surface is `thread/inject_items`; the SDK's typed methods are all one-liners over a generic `request`, so a `@patch` adds injection — and the create/append conveniences built on it — in the SDK's own shape.

In [ ]:
#| export
def codex_client(codex_home=None):
    "An `AsyncCodex` on the PATH `codex` binary, optionally with a redirected `CODEX_HOME`"
    return AsyncCodex(CodexConfig(codex_bin=shutil.which('codex'),
        env=dict(CODEX_HOME=str(codex_home)) if codex_home else None))

@patch
async def inject_items(self:AsyncThread, items):
    "Append raw Responses API `items` to this thread's model history"
    await self._codex._ensure_initialized()
    return await self._codex._client.request('thread/inject_items',
        dict(threadId=self.id, items=[obj2dict(o) for o in items]), response_model=ThreadInjectItemsResponse)

@patch
async def create_thread(self:AsyncCodex, items, cwd=None, **kwargs):
    "Start a persisted thread whose history is pre-filled with `items`"
    thread = await self.thread_start(cwd=str(Path(cwd).resolve()) if cwd else None, **kwargs)
    await thread.inject_items(items)
    return thread

@patch
async def append_thread(self:AsyncCodex, thread_id, items, **kwargs):
    "Resume `thread_id` and append raw Responses `items`"
    thread = await self.thread_resume(thread_id, **kwargs)
    await thread.inject_items(items)
    return thread

## Synthetic tool calls

In [ ]:
#| export
def tool_turn(prompt, name, arguments, output, answer):
    "A complete synthetic function-call turn"
    call = codex_call(name, arguments)
    return [codex_msg('user',prompt),call,codex_output(call['call_id'],output),codex_msg('assistant',answer)]

def custom_turn(prompt, name, arguments, output, answer):
    "A complete synthetic Codex custom-tool turn"
    call = codex_custom_call(name, arguments)
    return [codex_msg('user',prompt),call,codex_custom_output(call['call_id'],output),codex_msg('assistant',answer)]

In [ ]:
sample = tool_turn('Measure the flux.', 'flux_meter', {}, 'flux: 41.7 kilofinches', 'The flux is 41.7 kilofinches.')
kernel_sample = custom_turn('Evaluate the expression.', 'tools.mcp__clikernel__execute', dict(code='6*7'), '42', 'The result is 42.')
[(o['type'],o.get('name')) for o in kernel_sample]

## A sample session

The checked-in source rollout contains both examples. App-server created the file, then a second server resumed it and injected the custom call. No model ran.

In [ ]:
source_items = response_items(source_recs)
test('flux: 41.7 kilofinches', repr(source_items), in_)
test('custom_tool_call', source_items.attrgot('type'), in_)
len(source_items)

In [ ]:
#| eval: false
with tempfile.TemporaryDirectory() as d:
    home = Path(d)
    proj = home/'project'
    proj.mkdir()
    items = reid_items([*sample,*kernel_sample,codex_msg('user','x'*(70*1024))], 'live')
    async with codex_client(home) as codex: thread = await codex.create_thread(items, cwd=proj)
    async with codex_client(home) as codex: resumed = await codex.thread_resume(thread.id)
    test_eq(resumed.id, thread.id)
    print(thread.id, rollout_file(thread.id, home))

## Reading a session

`load_rollout` keeps every record. `active_items` is the reading path for model history; bookkeeping remains available for format investigation.

In [ ]:
test_eq(thread_id(source_recs), source_tid)
test_eq(len(active_items(source_recs)), len(response_items(source_recs)))
Counter(source_recs.attrgot('type'))

## Searching a session

In [ ]:
#| export
def _item_txts(o, skip=()):
    if isinstance(o, str): yield o
    elif isinstance(o, dict): yield from (t for k,v in o.items() if k not in skip for t in _item_txts(v,skip))
    elif is_listy(o): yield from (t for x in o for t in _item_txts(x,skip))

def item_txt(item):
    "Every readable string in a Responses item, joined"
    return '\n'.join(_item_txts(obj2dict(item), skip=('type','id','call_id','encrypted_content','role','status','internal_chat_message_metadata_passthrough')))

def item_role(item):
    "Conversation role of a Responses item"
    item = obj2dict(item)
    if item['type']=='message': return item['role']
    if item['type'] in ('function_call','custom_tool_call'): return 'assistant'
    if item['type'] in ('function_call_output','custom_tool_call_output'): return 'tool'
    return item['type']

def conv_items(items):
    "Conversation-bearing items, excluding reasoning and compaction"
    kinds = {'message','function_call','function_call_output','custom_tool_call','custom_tool_call_output'}
    return L(o for o in items if o.get('type') in kinds)

class ItemHits(list):
    "Search hits with one match-centered preview per line"
    def __repr__(self): return '\n'.join(f"{h.i:5} {h.role:10} {h.preview}" for h in self)

def item_search(
    pat, # Regex to find
    items, # Responses items
    maxlen=160, # Preview characters around the match
):
    "Search model-visible Responses items"
    items,res = conv_items(items),ItemHits()
    for i,o in enumerate(items):
        text = item_txt(o)
        if m := re.search(pat,text):
            h = maxlen//2
            s,e = max(0,m.start()-h),min(len(text),m.end()+h)
            preview = ('…' if s else '')+text[s:e].replace('\n',' ')+('…' if e<len(text) else '')
            res.append(AttrDict(i=i,role=item_role(o),preview=preview,item=o))
    res.items = items
    return res

def show_items(
    items, # Responses items
    mx=500, # Text characters per item
):
    "Readable model history for `items`"
    rows = []
    for o in conv_items(items):
        text = item_txt(o)
        if len(text)>mx: text = text[:mx]+f'…[+{len(text)-mx} chars]'
        rows.append(f"--- {item_role(o)}:{o['type']} ---\n{text}")
    return PrettyString('\n'.join(rows))

In [ ]:
hits = item_search(r'41\.7 kilofinches', source_items)
test_eq([h.role for h in hits], ['tool','assistant'])
hits

In [ ]:
show_items(hits.items[max(0,hits[0].i-1):hits[0].i+2], mx=120)

## Prompt history

Codex has no separate prompt-history JSONL. User messages remain in per-thread rollouts. `prompt_hist` scans them and returns the thread, project, timestamp, and text.

In [ ]:
#| export
class PromptHist(list):
    "Codex prompt-history rows, one line per user message"
    def __repr__(self):
        return '\n'.join(f"{h.ts[:16]} {Path(h.project).name:>12} {h.thread[:8]} {h.text.replace(chr(10),' ')[:100]}" for h in self)

def prompt_hist(
    project=None, # Only rollouts whose session cwd is this path
    since=None, # Only record timestamps at or after this ISO string
    pat=None, # Only prompt text matching this regex
    codex_home=CODEX_HOME, # Codex home containing sessions
):
    "User prompts retained in Codex rollouts, oldest first"
    rows = []
    for path in (Path(codex_home)/'sessions').rglob('*.jsonl'):
        recs = load_recs(path)
        meta = first((r.payload for r in recs if r.get('type')=='session_meta'), {})
        tid,cwd = meta.get('id'),meta.get('cwd','')
        if project and cwd!=str(Path(project).expanduser().resolve()): continue
        for r in recs:
            o = r.get('payload',{})
            if r.get('type')!='response_item' or o.get('type')!='message' or o.get('role')!='user': continue
            text = item_txt(o)
            if since and r.get('timestamp','')<since: continue
            if pat and not re.search(pat,text): continue
            rows.append(AttrDict(ts=r.get('timestamp',''),thread=tid,project=cwd,text=text,path=path))
    return PromptHist(sorted(rows,key=lambda x:x.ts))

## Curating a captured session

Reasoning items are large and replay does not require them. Tool inputs and outputs can also dominate a capture. The curation helpers return fresh dictionaries and keep custom calls executable after truncating their logical arguments.

In [ ]:
#| export
def strip_reasoning(items):
    "Drop Responses API reasoning items"
    return L(items).filter(lambda o:o.get('type')!='reasoning')

def _trunc_deep(o,mx):
    if isinstance(o,str): return o if len(o)<=mx else o[:mx]+f'…[+{len(o)-mx} chars]'
    if isinstance(o,dict): return {k:_trunc_deep(v,mx) for k,v in o.items()}
    if is_listy(o): return [_trunc_deep(x,mx) for x in o]
    return o

def trunc_tools(
    items, # Responses items
    mx=2000, # Maximum characters per string in tool arguments and results
):
    "Copies of `items` with tool arguments and outputs truncated"
    items = [obj2dict(o) for o in items]
    for i,o in enumerate(items):
        if o['type']=='function_call':
            try: args = json.loads(o.get('arguments') or '{}')
            except json.JSONDecodeError: continue
            o['arguments'] = json.dumps(_trunc_deep(args,mx))
        elif o['type']=='custom_tool_call' and (p:=parse_exec(o.get('input',''))):
            o['input'] = exec_input(p.name,_trunc_deep(p.arguments,mx))
        elif o['type'] in ('function_call_output','custom_tool_call_output'): o['output'] = _trunc_deep(o.get('output',''),mx)
    return L(items)

def curate_items(
    recs, # Captured rollout records
    key='', # Salt for deterministic IDs
    mx=None, # Optional tool string cap
):
    "Extract active history, drop reasoning, optionally truncate tools, and re-identify"
    items = strip_reasoning(active_items(recs))
    if mx is not None: items = trunc_tools(items,mx)
    return reid_items(items,key)

In [ ]:
curated = curate_items(source_recs, 'fixture')
test_eq(curated, curate_items(source_recs, 'fixture'))
calls = [o for o in curated if o['type'] in ('function_call','custom_tool_call')]
outs = [o for o in curated if o['type'] in ('function_call_output','custom_tool_call_output')]
test_eq([o['call_id'] for o in calls], [o['call_id'] for o in outs])
test_ne(calls[0]['call_id'], first(o for o in source_items if o['type'] in ('function_call','custom_tool_call'))['call_id'])

## Session names

Thread names live in Codex's thread index rather than as ad-hoc rollout records. App-server's `thread/name/set` operation updates that index. Thread IDs remain the portable resume reference.

In [ ]:
#| eval: false
async with codex_client() as codex: await (await codex.thread_resume(cur_thread())).set_name('oai fixture')

## From dialogs

An authored dialog converts through the same canonical history used by Claude. Normal tool calls become `function_call` items. A supported tool name beginning with `tools.` becomes a custom Codex `exec` item, so a dialog can show `tools.mcp__clikernel__execute(code=...)` while the resumed thread receives the current native JavaScript wrapper.

In [ ]:
#| export
def _customize_items(items):
    items,customs = list(items),set()
    for i,o in enumerate(items):
        if o['type']=='function_call' and o['name'].startswith('tools.'):
            customs.add(o['call_id'])
            items[i] = codex_custom_call(o['name'],json.loads(o['arguments']),o['call_id'])
        elif o['type']=='function_call_output' and o['call_id'] in customs: items[i] = codex_custom_output(o['call_id'],o['output'])
    return L(items)

def dlg2items(
    dlg, # Dialog ending with a prompt
    aim_info=None, # Model capability dict for media handling
):
    "Responses API items for `dlg`, including native custom `exec` calls"
    items = _customize_items(denorm_msgs(dlg2chat(dlg,aim_info)))
    raws,t = {},-1
    for m in dlg.messages:
        if m.msg_type==sprompt: t += 1
        elif m.msg_type==sraw and m.meta.get('item_kind'): raws.setdefault(t,[]).append(dict(m.meta['item']))
    if not raws: return items
    out,t = [],-1
    for o in items:
        if o['type']=='message' and o.get('role')=='user':
            out += raws.get(t,[])
            t += 1
        out.append(o)
    return L([*out,*raws.get(t,[])])

async def dlg2thread(
    dlg, # Dialog to convert
    cwd=None, # Project directory
    codex_home=None, # Codex home
    **kwargs, # Passed to thread/start
):
    "Create a persisted Codex thread whose history is `dlg`"
    async with CodexAppServer(codex_home=codex_home) as app:
        thread = await app.create_thread(reid_items(dlg2items(dlg),dlg.name),cwd=cwd,**kwargs)
    return thread.id

The familiar flux dialog still emits a paired ordinary function call and output.

In [ ]:
def tool_dtl(func,args,result):
    d = json.dumps(dict(id='call1',server=False,call=dict(function=func,arguments=args),result=result))
    return f"{tool_dtls_tag}\n<summary><code>{func}(...)</code></summary>\n\n```json\n{d}\n```\n\n</details>"

fdlg = Dialog(name='flux')
fdlg.mk_message('Measure the flux please.',msg_type=sprompt,
    output=f"Let me check.\n\n{tool_dtl('flux_meter',{},'flux: 41.7 kilofinches')}\n\nThe flux is 41.7 kilofinches.")
fitems = dlg2items(fdlg)
test_eq([o['type'] for o in fitems], ['message','message','function_call','function_call_output','message'])
test_eq(fitems[2]['call_id'],fitems[3]['call_id'])

Changing the tool name to `tools.mcp__clikernel__execute` exercises the native custom mapping used by current Codex sessions.

In [ ]:
kdlg = Dialog(name='kernel')
kdlg.mk_message('Evaluate it.',msg_type=sprompt,
    output=f"{tool_dtl('tools.mcp__clikernel__execute',{'code':'6*7'},'42')}\n\nThe result is 42.")
kitems = dlg2items(kdlg)
test_eq([o['type'] for o in kitems], ['message','custom_tool_call','custom_tool_call_output','message'])
test_eq(parse_exec(kitems[1]['input']).name, 'tools.mcp__clikernel__execute')

## Back to dialogs

`items2chat` normalizes messages and both tool-call families. Developer messages, reasoning, and native compaction are rollout context rather than editable turns; `items2dlg` preserves developer and compaction items as tagged raw messages so `dlg2items` can re-emit them.

In [ ]:
#| export
def _output_text(output):
    if isinstance(output,str): return output
    return '\n'.join(x.get('text','') for x in output if x.get('type') in ('input_text','output_text'))

def _msg_parts(content):
    parts = []
    for c in content:
        if c['type'] in ('input_text','output_text'): parts.append(Part(type=PartType.text,text=c['text']))
        elif c['type']=='input_image': parts.append(Part(type=PartType.input_image,text=c['image_url']))
        else: raise ValueError(f"unsupported message content: {c['type']}")
    return parts

def items2chat(
    items, # Responses API items
):
    "Canonical fastllm messages for editable conversation items"
    msgs,names,customs = [],{},set()
    for o in map(obj2dict,items):
        typ = o['type']
        if typ=='message':
            if o['role'] not in ('user','assistant'): continue
            msgs.append(Msg(role=o['role'],content=_msg_parts(o['content'])))
        elif typ in ('function_call','custom_tool_call'):
            if typ=='custom_tool_call':
                p = parse_exec(o['input'])
                name,args = (p.name,p.arguments) if p else ('exec',dict(input=o['input']))
                customs.add(o['call_id'])
            else: name,args = o['name'],json.loads(o.get('arguments') or '{}')
            names[o['call_id']] = name
            msgs.append(Msg(role='assistant',content=[Part(type=PartType.tool_use,
                data=dict(id=o['call_id'],name=name,arguments=args))]))
        elif typ in ('function_call_output','custom_tool_call_output'):
            cid = o['call_id']
            msgs.append(Msg(role='tool',content=[Part(type=PartType.tool_result,text=_output_text(o.get('output','')),
                data=dict(id=cid,name=names.get(cid),custom=cid in customs))]))
        elif typ in ('reasoning','compaction'): continue
        else: raise ValueError(f'unsupported response item: {typ}')
    return msgs

def chat2dlg(
    msgs, # Canonical messages
    name, # Dialog name
    cls=Dialog, # Dialog class
    mx=2000, # Maximum rendered tool string length; None disables truncation
):
    "A dialog for canonical messages, one prompt per user turn"
    dlg,turns = cls(name=name),[]
    for m in msgs:
        if m.role=='user': turns.append((m,[]))
        elif turns: turns[-1][1].append(m)
        else: raise ValueError('msgs must start with a user message')
    for u,replies in turns:
        segs,atts = [],[]
        for p in u.content:
            if p.type==PartType.text: segs.append(p.text)
            elif p.type==PartType.input_image:
                mime,data = data_url(p.text)
                atts.append(Attachment(base64.b64decode(data),mime))
                segs.append(f'![](attachment:{atts[-1].id})')
            else: raise ValueError(f'unsupported user part: {p.type}')
        dlg.mk_message('\n\n'.join(segs),msg_type=sprompt,output=hist2fmt(replies,mx=mx),attachments=atts)
    return dlg

def items2dlg(
    items, # Responses API items
    name='thread', # Dialog name
    mx=2000, # Maximum rendered tool string length
):
    "Editable dialog for `items`, preserving developer and compaction items as tagged raws"
    items = L(obj2dict(o) for o in items)
    dlg = chat2dlg(items2chat(items),name,mx=mx)
    prompts = [m for m in dlg.messages if m.msg_type==sprompt]
    raws,t = {},-1
    for o in items:
        if o['type']=='message' and o.get('role')=='user': t += 1
        elif o['type']=='compaction' or (o['type']=='message' and o.get('role') not in ('user','assistant')):
            raws.setdefault(t,[]).append(o)
    for t,group in reversed(list(raws.items())):
        before = prompts[t+1] if t+1<len(prompts) else None
        for o in reversed(group):
            kw = dict(before=before) if before else dict(idx=-1)
            before = dlg.mk_message('',msg_type=sraw,meta=dict(item_kind=o['type'],item=o),**kw)
    return dlg

def thread2dlg(
    thread_id=None, # Thread UUID; current thread if None
    codex_home=CODEX_HOME, # Codex home
    name=None, # Dialog name
    mx=2000, # Maximum rendered tool string length
    recs=None, # Already-loaded rollout records
):
    "The active model history of a Codex thread as a dialog"
    thread_id = thread_id or cur_thread()
    recs = load_rollout(thread_id,codex_home) if recs is None else recs
    return items2dlg(active_items(recs),name or thread_id or 'thread',mx)

In [ ]:
round_dlg = items2dlg(kitems,'kernel back',mx=None)
back = dlg2items(round_dlg)
test_eq([o['type'] for o in back], [o['type'] for o in kitems])
test_eq(parse_exec(back[1]['input']).arguments['code'], '6*7')
test_eq(back[2]['call_id'],back[1]['call_id'])

In [ ]:
source_dlg = thread2dlg(name='source fixture',recs=source_recs)
test('41.7 kilofinches','\n'.join(m.content+m.ai_res for m in source_dlg.messages),in_)
source_dlg

## Inside a live session

`CODEX_THREAD_ID` is available to command processes started by Codex, but not to the long-lived MCP server. These cells inspect the running thread when the environment provides an id and otherwise use the checked-in fixture.

In [ ]:
live_tid = cur_thread()
live_recs = load_rollout(live_tid) if live_tid and rollout_file(live_tid) else source_recs
live_tid or source_tid,Counter(live_recs.attrgot('type'))

## Resolving session references

Codex names are index metadata, while rollout filenames are keyed by UUID. File-level tools therefore resolve the current or explicit UUID. Name lookup belongs to app-server clients rather than filesystem heuristics.

In [ ]:
#| export
def resolve_thread(
    ref=None, # Thread UUID; current thread if None
    codex_home=CODEX_HOME, # Codex home
):
    "Resolve a thread UUID to `(thread_id, rollout_path)`"
    ref = ref or cur_thread()
    path = rollout_file(ref,codex_home)
    if not path: raise FileNotFoundError(f'No Codex thread {ref!r}')
    return ref,path

In [ ]:
with expect_fail(FileNotFoundError): resolve_thread('missing-thread',tempfile.mkdtemp())

## Recognizing earlier compactions

`split_compaction` does not infer a boundary from prose. It reads the latest native `compacted` record, preserving its encrypted compaction item and replacement messages exactly, then returns only response items appended later.

In [ ]:
prior,new = split_compaction(compact_recs)
test_eq(prior[-1]['type'],'compaction')
test_eq(item_txt(new[0]),'Continue the work.')
test_eq(response_items(compact_recs)[0]['content'][0]['text'],'Old request')

## Native compaction

Codex performs compaction itself through `thread/compact/start`. The result is an encrypted replacement history, which llmsurgery preserves exactly rather than trying to decode. The operation calls a model and is intentionally a live example; the next section writes the same record shape with a readable compact-DSL document instead.

In [ ]:
#| eval: false
async with codex_client() as codex:
    thread = await codex.thread_resume(cur_thread())
    await thread.compact()

## Synthetic compaction

The compact DSL describes conversation content; this section writes it in Codex's own record format. A synthetic compaction is one `compacted` record appended to the rollout while Codex is closed: its `replacement_history` carries the context items the DSL cannot express, ending with one user message holding the compact document. Codex rebuilds history from the latest `compacted` record, so a resumed thread starts from the document instead of the full transcript.

Real native records ground the mirrored shape. Alongside `message` (blank in practice) and `replacement_history`, current payloads carry `window_number` (how many compactions so far) and `first`/`previous`/`window_id` context-window UUIDs; older rollouts have fewer fields, and Codex still reads them. The pinning rule is visible in the replacement roles: every prior `user` and `developer` message is kept verbatim, in order, then one encrypted `compaction` item stands in for everything else. On a machine with compacted Codex threads, the newest one shows both:

In [ ]:
#| eval: false
paths = sorted((CODEX_HOME/'sessions').rglob('*.jsonl'), reverse=True)
c = first(r for p in paths for r in load_recs(p) if r.get('type')=='compacted')
list(c.payload), [(o.get('type'),o.get('role')) for o in c.payload.replacement_history]

In [ ]:
#| export
def _ts(): return datetime.now(timezone.utc).isoformat(timespec='milliseconds').replace('+00:00','Z')

def _ctx_item(o):
    "Is `o` context the compact DSL can't express: a non-conversation message, or a compaction item?"
    return o.get('type')=='compaction' or (o.get('type')=='message' and o.get('role') not in ('user','assistant'))

def _split_synthetic(items):
    "Items to carry forward, and the DSL body of a trailing synthetic summary"
    o = items[-1] if items else {}
    if o.get('type')=='message' and '\n\n## Conversation\n\n' in (t := _output_text(o.get('content',''))):
        return L(items[:-1]),compact_body(t)
    return L(items),''

Where Codex pins every user and developer message, the synthetic replacement pins only what the DSL cannot render — developer and system messages, and prior `compaction` items — since the document itself keeps user text at high fidelity (`user_toks=1000`). `_split_synthetic` is the incremental hook: a trailing synthetic summary is recognized by its document body, extracted, and re-joined with newly-compacted turns, while a native (encrypted) prior is carried forward untouched.

In [ ]:
keep,body = _split_synthetic(prior)
test_eq((keep,body), (prior,''))
doc_msg = codex_msg('user', compact_content('§ Hi. §', '/tmp/t.jsonl'))
keep,body = _split_synthetic(prior+[doc_msg])
test_eq((list(keep),body), (list(prior),'§ Hi. §'))
assert _ctx_item(dict(type='message',role='developer')) and not _ctx_item(doc_msg)

In [ ]:
#| export
def prepare_compaction(ref=None, codex_home=CODEX_HOME, policy=compact_policy, enc=None):
    "Prepare an incremental synthetic compaction without writing it"
    tid,path = resolve_thread(ref, codex_home)
    recs = load_recs(path)
    prior,new = split_compaction(recs)
    keep,prior_body = _split_synthetic(prior)
    msgs = items2chat(new)
    if not msgs: raise ValueError(f'No new items to compact in {tid}')
    new_chat = compact_chat(msgs, enc=enc, last_n=5, **policy)
    new_full = compact_chat(msgs, enc=enc, **{k:10**9 for k in policy})
    chat,full = join_compacts(prior_body,new_chat),join_compacts(prior_body,new_full)
    content = compact_content(chat, path)
    comps = [r.payload for r in recs if r.get('type')=='compacted']
    prev = comps[-1].get('window_id') if comps else nested_idx(recs[0], 'payload', 'context_window', 'window_id')
    first = comps[0].get('first_window_id') if comps else prev
    payload = dict(message='', replacement_history=list(keep + L(new).filter(_ctx_item) + [codex_msg('user', content)]),
        window_number=len(comps)+1, first_window_id=first, previous_window_id=prev, window_id=str(uuid.uuid4()))
    rec = dict(timestamp=_ts(), type='compacted', payload=payload)
    return AttrDict(tid=tid,path=path,rec=rec,prior=prior_body,new_chat=new_chat,chat=chat,full=full,content=content,
        pre_toks=len_toks(full,enc),post_toks=len_toks(content,enc))

`prepare_compaction` is read-only and mirrors `llmsurgery.ant`: the prior synthetic body (if any) joins the newly-compacted items, the final five messages stay untruncated, and reasoning items vanish just as native compaction drops them. The window fields follow the current native semantics — `window_number` counts compactions, `previous_window_id` is the window being replaced (the thread's original from `session_meta` when this is the first), and a fresh `window_id` starts the next one. Preparing against a copy of the source fixture pins its three developer messages (permissions and agent instructions) ahead of the document.

In [ ]:
home = Path(tempfile.mkdtemp())
day = home/'sessions'/'2026'/'07'/'19'
day.mkdir(parents=True)
(day/f'rollout-2026-07-19T05-30-38-{source_tid}.jsonl').write_bytes(source_path.read_bytes())
compaction = prepare_compaction(source_tid, home)
pl = compaction.rec['payload']
test_eq([o['role'] for o in pl['replacement_history']], ['developer','developer','developer','user'])
test_eq((pl['window_number'],pl['previous_window_id']), (1,'019f78da-e13e-72a2-9c10-b003c05ac899'))
test('41.7 kilofinches', compaction.chat, in_)
test('## Conversation', compaction.content, in_)
test_eq(compaction.prior, '')

In [ ]:
#| export
def append_compaction(compaction):
    "Append a prepared compaction record to its thread's rollout"
    with Path(compaction.path).open('a') as f: f.write(json.dumps(obj2dict(compaction.rec))+'\n')
    return compaction.path

def compact_session(ref=None, codex_home=CODEX_HOME, policy=compact_policy, enc=None):
    "Generate and append a synthetic thread compaction"
    compaction = prepare_compaction(ref, codex_home, policy, enc)
    append_compaction(compaction)
    return compaction

Appending happens while Codex is closed, like `ant.append_compaction`; the rollout is append-only, so one record establishes the new active history, and thread names live in the app-state index rather than the rollout, so nothing needs restoring. After the append, the whole active history is the replacement, and the document body round-trips through `_split_synthetic`:

In [ ]:
append_compaction(compaction)
prior2,new2 = split_compaction(load_rollout(source_tid, home))
test_eq((len(new2),_split_synthetic(prior2)[1]), (0,compaction.chat))

Repeated compaction is incremental, as in `ant`: the prior document body is preserved verbatim while only records added after the last compaction are newly compacted, and the window fields chain — `previous_window_id` picks up the last compaction's `window_id`:

In [ ]:
more = [dict(timestamp=_ts(), type='response_item', payload=p)
    for p in (codex_msg('user','Also measure the spam.'), codex_msg('assistant','Spam is at 7 decibels.'))]
with compaction.path.open('a') as f: f.writelines(json.dumps(o)+'\n' for o in more)
c2 = compact_session(source_tid, home)
test_eq((c2.prior,c2.rec['payload']['window_number']), (compaction.chat,2))
test_eq(c2.rec['payload']['previous_window_id'], pl['window_id'])
test('Spam is at 7 decibels', c2.new_chat, in_)
test_eq(split_compaction(load_rollout(source_tid, home))[1], L())

The acceptance check is live: create a thread through app-server, synthetically compact it offline, and resume — Codex parses the appended record and rebuilds the active history from the pinned context plus the document.

In [ ]:
#| eval: false
with tempfile.TemporaryDirectory() as d:
    lhome,proj = Path(d),Path(d)/'project'
    proj.mkdir()
    items = reid_items([*sample,*kernel_sample], 'synthetic')
    async with codex_client(lhome) as codex: thread = await codex.create_thread(items, cwd=proj)
    compact_session(thread.id, lhome)
    async with codex_client(lhome) as codex: resumed = await codex.thread_resume(thread.id)
    test_eq(resumed.id, thread.id)
    print(show_items(active_items(load_rollout(thread.id, lhome)), mx=80))

## Cleanup

Temporary app-server examples use their own Codex home and disappear with the temporary directory. Checked-in fixtures are package data and remain unchanged.

## Headless runs with `codex exec`

Everything above works on rollout files; `codex exec` is how new ones get made without a TUI: one prompt in, an agent run out. The prompt comes from argv or stdin, `-C` sets the working root (a git repo, unless `--skip-git-repo-check`), and `-m` picks the model. Any `config.toml` key can be overridden per run with `-c key=value`, the value parsed as TOML; `model_reasoning_effort` and `approval_policy` are the useful ones here. `--ephemeral` skips writing a session rollout, and `--ignore-user-config` skips `~/.codex/config.toml`, though auth, skills, and `.rules` files still load. A ChatGPT subscription login works from headless runs.

For scripting, `--json` turns stdout into a JSONL event stream: `thread.started`, `turn.started`, `item.started`/`item.completed` (item types include `agent_message`, `command_execution` with the exact command and exit code, `file_change`, and `mcp_tool_call`), then `turn.completed` with token usage. `-o path` also writes the final agent message to a file, and `--output-schema` makes it conform to a JSON Schema. A piped stdin is appended to the prompt as a `<stdin>` block, so automation should redirect stdin from `/dev/null`.

In [ ]:
#| eval: false
!codex exec --skip-git-repo-check -C /tmp/sandbox -m gpt-5.6-luna \
    -c model_reasoning_effort=low --json 'Create hello.txt containing: hi' > events.jsonl

The permission model is the inverse of Claude Code's. There is no per-command allowlist inside the sandbox. Instead `-s read-only`, `-s workspace-write`, or `-s danger-full-access` selects an OS-level sandbox (Seatbelt on macOS) bounding what any command may touch: `workspace-write` permits writes only under the working root and temp directories, with network off by default, and within those bounds the agent runs whatever commands it likes.

Two mechanisms control crossing that boundary. First, the model can request that a command run outside the sandbox, which normally prompts the user; `-c approval_policy=never` denies every such request, making the sandbox a hard boundary (verified: under `workspace-write` a requested write outside the workspace was refused, and the run reported it could not comply). Second, execpolicy rules, Starlark files like this:

```python
prefix_rule(
    pattern = ["/path/to/step1.sh"],
    decision = "allow",
    justification = "prepared command for automation",
)
```

Rules match argv prefixes. `decision` is `allow`, `prompt`, or `forbidden`, and the most restrictive match wins. An `allow` match runs the command outside the sandbox with no prompt, but only when every command in the invocation matches an allow rule: a `bash -lc` wrapper whose script is a plain linear chain (`&&`, `||`, `;`, `|`, with no redirects, expansions, or assignments) is split with tree-sitter and each piece checked separately, while anything fancier is checked as one opaque invocation that matches nothing. Verified live under `-s read-only -c approval_policy=never`: the allowed script's writes succeeded even though the model wrapped it in `bash -lc`, and `step1.sh && echo x >> file.txt` ran entirely inside the read-only sandbox, where both writes were denied.

That combination is a locked-subagent recipe: read-only sandbox, escalation denied, allow rules naming exactly the scripts the child may run for real. The child reads and reasons freely, and the only state changes it can make are the prepared commands.

Getting allow rules into a run has traps, each of which cost a debugging round:

- No flag takes a rules file. Allow rules load only from a `rules/` folder of an active config layer: `~/.codex/rules/*.rules` (applies to every run) or `<repo>/.codex/rules/*.rules` (loads only when the project is trusted). The config key `rules.prefix_rules` cannot substitute: it accepts only `prompt` and `forbidden`.
- Trust can be granted per run with `-c projects.<path>.trust_level=trusted`, but only for paths containing no dots: the `-c` parser splits the key on every `.` and keeps quote characters literally, so the TOML-style `projects."/path"` spelling silently stores a wrong key and the project rules never load.
- Project detection starts from the process cwd, and the default root marker is a `.git` directory; its existence is enough, and `project_root_markers` makes it configurable.
- `codex execpolicy check --rules file.rules cmd args...` evaluates a command against a rules file offline and prints the decision as JSON, e.g. `{"decision":"allow","matchedRules":[...]}`. Test rules this way before relying on them, since a rules file that fails to load is a silent no-op.

Two costs to know about. Every run reloads the user's skills and `AGENTS.md`, even under `--ignore-user-config`: tens of thousands of input tokens per call, though mostly cache hits. And there is no Python SDK; the TypeScript `@openai/codex-sdk` and an MCP server mode exist, but from Python you spawn the CLI per task and parse the JSONL.

In [ ]:
#| eval: false
!codex exec --skip-git-repo-check -C /tmp/sandbox -s read-only -c approval_policy=never \
    -c projects./tmp/sandbox.trust_level=trusted --json \
    'Run exactly this command: /tmp/sandbox/step1.sh then review its output' > run.jsonl

## Export -

In [ ]:
#| hide
#| eval: false
import nbdev; nbdev.nbdev_export()